## Setup 

In [ ]:
source(file.path("~/workspace/pipelines/snt_dhis2_formatting/utils/snt_dhis2_formatting.r"))
snt_paths <- init_snt_workspace(
    snt_pipeline_name="snt_dhis2_formatting",
    packages=c("lubridate", "arrow", "dplyr", "tidyr", "stringr", "stringi", "jsonlite", "httr", "sf", "rmapshaper", "glue")) # "zoo"

# Load config
config_json <- load_snt_config(file.path(snt_paths$CONFIG_PATH, "SNT_config.json"))

# Save config variables
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
ADMIN_1 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_1)
ADMIN_2 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_2)
dataset_name <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_EXTRACTS

### Load DHIS2 shapes data

-Load DHIS2 shapes from latest dataset version 


In [ ]:
dhis2_data <- load_dataset_file(dataset_name, paste0(COUNTRY_CODE, "_dhis2_raw_shapes.parquet"))
head(dhis2_data %>% select(-GEOMETRY), 3)

## SNT Shapes formatting

### Administrative columns selection

In [ ]:
# Set administrative column names
adm_1_id_col       <- gsub("_NAME", "_ID", ADMIN_1)
adm_1_name_col     <- ADMIN_1
adm_1_lvl_name_col <- gsub("LEVEL_", "ORG_LEVEL_", ADMIN_1)
adm_2_id_col       <- gsub("_NAME", "_ID", ADMIN_2)
adm_2_name_col     <- ADMIN_2
adm_2_lvl_name_col <- gsub("LEVEL_", "ORG_LEVEL_", ADMIN_2)

# Check mandatory administrative columns
if (!adm_1_id_col %in% colnames(dhis2_data)) stop("[ERROR]: Mandatory column '{adm_1_id_col}' not found.")
if (!adm_1_name_col %in% colnames(dhis2_data)) stop("[ERROR]: Mandatory column '{adm_1_name_col}' not found.")
if (!adm_2_id_col %in% colnames(dhis2_data)) stop("[ERROR]: Mandatory column '{adm_2_id_col}' not found.")
if (!adm_2_name_col %in% colnames(dhis2_data)) stop("[ERROR]: Mandatory column '{adm_2_name_col}' not found.")
admin_columns <- c(adm_1_id_col, adm_1_name_col, adm_2_id_col, adm_2_name_col)

# Check level name columns (if present)
has_adm1_level <- FALSE
has_adm2_level <- FALSE
if (adm_1_lvl_name_col %in% colnames(dhis2_data)) {
    has_adm1_level <- TRUE    
    admin_columns <- c(admin_columns, adm_1_lvl_name_col)
}
if (adm_2_lvl_name_col %in% colnames(dhis2_data)) {
    has_adm2_level <- TRUE
    admin_columns <- c(admin_columns, adm_2_lvl_name_col)
}

log_msg(glue("Administrative columns selection: {paste(admin_columns, collapse=', ')}"))

### Format table

In [ ]:
# Data selection
shapes_data <- dhis2_data[, c(admin_columns, "GEOMETRY")]

# Clean strings for admin 1 and admin 2 (format_names() in snt_utils.r)
shapes_data[[adm_1_name_col]] <- format_names(shapes_data[[adm_1_name_col]]) 
shapes_data[[adm_2_name_col]] <- format_names(shapes_data[[adm_2_name_col]])
if (has_adm1_level) shapes_data[[adm_1_lvl_name_col]] <- format_names(shapes_data[[adm_1_lvl_name_col]])
if (has_adm2_level) shapes_data[[adm_2_lvl_name_col]] <- format_names(shapes_data[[adm_2_lvl_name_col]])

# Select and Rename columns
shapes_data <- shapes_data %>%
    select(
        ADM1_NAME = !!sym(adm_1_name_col),
        ADM1_ID = !!sym(adm_1_id_col),
        if (has_adm1_level) c(ADM1_LEVEL_NAME = adm_1_lvl_name_col),
        ADM2_NAME = !!sym(adm_2_name_col),
        ADM2_ID = !!sym(adm_2_id_col),
        if (has_adm2_level) c(ADM2_LEVEL_NAME = adm_2_lvl_name_col),
        GEOMETRY
    )

# Select metadata columns only for checking (preview)
preview_cols <- intersect(
    c("ADM1_ID", "ADM1_NAME", "ADM1_LEVEL_NAME", "ADM2_ID", "ADM2_NAME", "ADM2_LEVEL_NAME"),
    colnames(shapes_data)
)
head(shapes_data[, preview_cols])

### Transform shapes data to valid geojson 

In [ ]:
# Safe Convert geometry column from GeoJSON to 'sf' (simple feature geometry)
# Ignore wrong and empty geometries.
shapes_data_sf <- geojson_to_sf(shapes_data)

# Simplify shapes (reduce details)
shapes_sf_simplified <- simplify_geometries(shapes_data_sf, keep = 0.05)
cat(glue("Geometries simplified, original size: {round(object.size(shapes_data_sf)/1000000,3)} MB new size: {round(object.size(shapes_sf_simplified)/1000000,3)} MB"))

# Rename column to "GEOMETRY"
current_geom <- attr(shapes_sf_simplified, "sf_column")
shapes_sf_simplified <- shapes_sf_simplified %>% rename(GEOMETRY = !!sym(current_geom))

In [ ]:
# Validation check
err_vertices <- st_is_valid(shapes_sf_simplified, reason = TRUE)
if (length(err_vertices[err_vertices != 'Valid Geometry']) > 0) {
    log_msg("Invalid shapes found, please check the execution of: pipelines/snt_dhis2_formatting/papermill_outputs/snt_dhis2_formatting_shapes_OUTPUT_*.ipynb", "warning")
}

cat("Dimensions:", nrow(shapes_sf_simplified), "rows x", ncol(shapes_sf_simplified), "columns\n")
head(as.data.frame(shapes_sf_simplified))

In [ ]:
# ADM 2 level check!
plot(shapes_sf_simplified[, "ADM2_ID"])

### Output formatted shapes data

In [ ]:
# save file path
FORMATTED_DATA_PATH <- file.path(snt_paths$DATA_PATH, "dhis2", "extracts_formatted")
file_path <- file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, "_shapes.geojson"))

# save geojson
sf::st_write(shapes_sf_simplified, dsn = file_path, layer = file_path, delete_dsn = TRUE)

# log
log_msg(glue("Shapes data saved under: {file_path}"))

### Data Summary 

In [ ]:
# Data summary
print(summary(shapes_data_sf))